**Building conversational Chains: Messages and Tools**

- Fun part agents that can handle messages and external tools. Combines four key concepts:
    * chat messages
    * chat models
    * binding tools to LLMx
    * executing tools with the 

*Messages*

- chat models use sequence of "messages" that capture the different roles in a conversation.

- LangChain provides message types like:
    * HumanMessage: What the user said
    * AIMessage: What the AI says
    * SystemMessage: Instructions for the AI
    * ToolMessage: Output of a tool call

- Each message has content, and optional name(who sent it) and "response_metadata"

In [4]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages=[AIMessage(content=f'So you were researching about LangChain? ', name='Model')]
messages.extend([HumanMessage(content=f'Yes', name='Allan')])
messages.extend([AIMessage(content=f'What would you want to learn?', name='Model')])
messages.extend([HumanMessage(content=f'I want to learn about stategrapph in LangGraph(define it in one sentence)', name="Allan")])

for msg in messages:
    msg.pretty_print()

# We will take a list of messages and pass it to our chatModel


================================== Ai Message ==================================
Name: Model

So you were researching about LangChain?
================================ Human Message =================================
Name: Allan

Yes
================================== Ai Message ==================================
Name: Model

What would you want to learn?
================================ Human Message =================================
Name: Allan

I want to learn about stategrapph in LangGraph(define it in one sentence)


**ChatModel**

- A ChatModel like OpenAI, llama takes a list of these messages as input and generates an AIMessage as output.


In [5]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

#load environment variables: API keys
load_dotenv()

#initialize our model
llm=ChatGroq(
    temperature=0.3,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

# pass our list of messages to the LLM
result=llm.invoke(messages)

#output
print(f"AI: {result.content}")

AI: A StateGraph in LangChain is a data structure that represents the current state of a conversation or application, storing and managing the context, variables, and other relevant information to facilitate seamless interactions and decision-making.


**Tool Calling & Tools**

- Tools are crucial whenever we want our agent to interact with the outside world(Reason and Take Action)

- Examples:
    * Search the web
    * Access Databases
    * Perform Calculations

- Modern LLMs support tool calling(Structured data)


*Steps in tool Calling*

- Tool Creation: Use the @tool decorator to create a tool. A tool is an association between python and its schema(describes its purpose and required inputs).

- Tool Binding: The tool needs to be connected to a model that supports tool calling. Provides model with tool awareness and associated input schema required by the tool.

- Tool Calling: A model calls a tool based on the user's input and conversation history. Ensures response conforms to the tool's input schema---> providing necessary arguments

- Tool Execution: Tool executed using the arguments provided by the model. Output returned to the model to be used in next response.


In [8]:
from langchain_core.tools import tool


#---1. Creating tool
@tool      #python decorator
def multiply(a: int, b: int)->int:
    "Multiply a and b"   #python docstring---> Provides information to the modle to what the tool does
    return a*b

#---2. Binding tool to the model
# Binding tool with LLM model
llm_with_tool=llm.bind_tools([multiply])

#---3. Invoke tool usage bythe LLM
tool_call=llm_with_tool.invoke("I have 600 per day, How much will I have in 115 days?")
print(tool_call)

print("\n =====Below Shows Tool Usage well=====")

# Above shows no content in AI message but there is a tool call
print(tool_call.additional_kwargs['tool_calls'])  # shows the tool call structure.



content='' additional_kwargs={'tool_calls': [{'id': '3kdf3nekw', 'function': {'arguments': '{"a":600,"b":115}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 231, 'total_tokens': 250, 'completion_time': 0.036799584, 'completion_tokens_details': None, 'prompt_time': 0.012350156, 'prompt_tokens_details': None, 'queue_time': 0.047280731, 'total_time': 0.04914974}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d2e27-6435-7792-93e2-7c135623c913-0' tool_calls=[{'name': 'multiply', 'args': {'a': 600, 'b': 115}, 'id': '3kdf3nekw', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 231, 'output_tokens': 19, 'total_tokens': 250}

 =====Below Shows Tool Usage well=====
[{'id': '3kdf3nekw', 'function': {'arguments': '{"a":600,"b":115}', 'name': 'mult